In [53]:
from tcgdexsdk import TCGdex, Language
from bs4 import BeautifulSoup
import requests
tcgdex_fr = TCGdex(language=Language.FR)
tcgdex_en = TCGdex(language=Language.EN)

In [54]:
selected_attributes = [
    "illustrator",
    "rarity",
    "localId"
]

async def get_card_details(card_id: str, selected_attributes: list[str] = selected_attributes, tcgdex: TCGdex = tcgdex_fr):
    card = await tcgdex.card.get(card_id)
    print({attr: getattr(card, attr) for attr in selected_attributes})
    card_url = card.get_image_url(quality="low", extension="png")
    print(f"Card image URL: {card_url}")

    print(vars(card))

await get_card_details("swsh3-136")

{'illustrator': 'tetsuya koizumi', 'rarity': 'Peu Commune', 'localId': '136'}
Card image URL: https://assets.tcgdex.net/fr/swsh/swsh3/136/low.png
{'illustrator': 'tetsuya koizumi', 'rarity': 'Peu Commune', 'category': 'Pokémon', 'variants': CardVariants(normal=True, reverse=True, holo=False, firstEdition=False, wPromo=False), 'set': SetResume(sdk=<tcgdexsdk.tcgdex.TCGdex object at 0x10541f110>, id='swsh3', name='Ténèbres Embrasées', logo='https://assets.tcgdex.net/fr/swsh/swsh3/logo', symbol='https://assets.tcgdex.net/univ/swsh/swsh3/symbol', cardCount=SetCardCountResume(total=201, official=189)), 'dexId': [162], 'hp': 110, 'types': ['Incolore'], 'evolveFrom': 'Fouinette', 'description': None, 'level': None, 'stage': 'Niveau 1', 'suffix': None, 'item': None, 'abilities': None, 'attacks': [CardAttack(name='Mode Cool', cost=['Incolore'], effect='Piochez 3 cartes.', damage=None), CardAttack(name='Éclate-Queue', cost=['Incolore'], effect="Lancez une pièce. Si c'est pile, cette attaque ne f

In [ ]:
base_url = "https://www.pullrates.gg/sets"

def get_pull_rates(soup: BeautifulSoup, card_no: list[str]):
    all_anchors = soup.find_all('a')

    filtered_anchors = []

    for anchor in all_anchors:
        aria_label = anchor.get("aria-label")
        if aria_label and "pull rates" in aria_label:
            href = anchor.get("href")
            card_number = int(href.split("-")[-1]) if href else None
            if card_number in card_no:
                filtered_anchors.append(anchor)

    if not filtered_anchors:
        print(f"No pull rates found for card numbers {card_no}")
        return None

    pull_rates = []

    for anchor in filtered_anchors:
        pull_rate_spans = anchor.find_all("span")
        if len(pull_rate_spans) < 2:
            print(f"Unexpected format for pull rate information for card numbers {card_no} in set.")
            return None
        pull_rate_text = pull_rate_spans[0].text.strip()
        pull_rate = int(pull_rate_text.split(":")[-1].strip()) if ":" in pull_rate_text else None
        pull_rates.append(pull_rate)

    return pull_rates

def get_booster_price(soup: BeautifulSoup):
    booster_price = int(soup.find_all("p", class_="text-3xl font-bold tabular text-accent")[0].text.strip()[:-1])
    return booster_price

def calculate_set_popularity(soup: BeautifulSoup, booster_price: int, release_date: str):
    popularity = 0

    return popularity


async def get_set_details(set_id: str, tcgdex: TCGdex = tcgdex_en):
    set_data = await tcgdex.set.get(set_id)
    set_name = set_data.name
    release_date = set_data.releaseDate

    set_name_url_format = set_name.lower().replace(" ", "-")
    url = f"{base_url}/{set_name_url_format}"
    response = requests.get(url)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    pull_rates = get_pull_rates(soup, [int(card.localId) for card in set_data.cards])
    booster_price = get_booster_price(soup)
    set_popularity = calculate_set_popularity(soup, booster_price, release_date)

    print(f"Set Name: {set_name}")

await get_set_details("swsh3")

8.00 
Set Name: Darkness Ablaze
